In [1]:
# Cell 1: Imports
import os
import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader, random_split
import glob

from gan_utils.io import H5ImageDataset
from gan.model import Generator, Discriminator, Encoder, DiscriminatorFeatures
from gan.trainer import Trainer
from gan.trainer_encoder import EncoderTrainer

import matplotlib.pyplot as plt




In [2]:
CONFIG = {
    "h5_path": "/projects/standard/fortson/shared/raj00075/anomalies.h5",
    "parquet_path": "/projects/standard/fortson/shared/raj00075/mask_labels_anomaly.gzip",
    "outdir": "/projects/standard/fortson/shared/raj00075/outputs",
    "gan_ckpt_dir": "/projects/standard/fortson/shared/raj00075/outputs/gan_checkpoints",
    "encoder_ckpt_dir": "/projects/standard/fortson/shared/raj00075/outputs/encoder_checkpoints_new",

    "batch_size": 32,
    "workers": 8,
    "val_split": 0.1,

    "image_size": 256,
    "n_z": 256,
    
    "n_layers": 6,

    "encoder_epochs": 200,
    "g_lr": 1e-5,
    "d_lr": 1e-5,
    "enc_lr": 1e-5,

    "seed": 42,
    "device": "cuda" if torch.cuda.is_available() else "cpu",
}


In [3]:
torch.manual_seed(CONFIG["seed"])
np.random.seed(CONFIG["seed"])
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(CONFIG["seed"])

os.makedirs(CONFIG["encoder_ckpt_dir"], exist_ok=True)


In [4]:
df = pd.read_parquet(CONFIG["parquet_path"])
dataset = H5ImageDataset(df, CONFIG["h5_path"], transforms=None)

n_total = len(dataset)
n_val = int(CONFIG["val_split"] * n_total)
n_train = n_total - n_val

g = torch.Generator().manual_seed(CONFIG["seed"])
train_ds, val_ds = random_split(dataset, [n_train, n_val], generator=g)

train_loader = DataLoader(
    train_ds,
    batch_size=CONFIG["batch_size"],
    shuffle=True,
    num_workers=CONFIG["workers"],
    pin_memory=True,
    persistent_workers=True,
)

val_loader = DataLoader(
    val_ds,
    batch_size=CONFIG["batch_size"],
    shuffle=False,
    num_workers=CONFIG["workers"],
    pin_memory=True,
    persistent_workers=True,
)


In [5]:
gen = Generator(
    n_z=CONFIG["n_z"],
    input_filt=CONFIG["image_size"],
    final_size=CONFIG["image_size"],
    n_layers=CONFIG["n_layers"],
    out_channels=3,
    norm=False,
    pool=False,
)

disc = Discriminator(
    in_channels=3,
    n_layers=CONFIG["n_layers"],
    input_size=CONFIG["image_size"],
    norm=False,
    pool=False,
)

enc = Encoder(
    in_channels=3,
    n_z=CONFIG["n_z"],
    n_layers=CONFIG["n_layers"],
    input_size=CONFIG["image_size"],
    norm=False,
    pool=False,
)

disc_feat = DiscriminatorFeatures(
    in_channels=3,
    n_layers=CONFIG["n_layers"],
    input_size=CONFIG["image_size"],
    norm=False,
    pool=False,
)


In [6]:
gen = gen.to(CONFIG["device"]).eval()
disc = disc.to(CONFIG["device"]).eval()
enc = enc.to(CONFIG["device"])

gen.load_state_dict(torch.load(f"{CONFIG['gan_ckpt_dir']}/generator_final.pth"))
disc.load_state_dict(torch.load(f"{CONFIG['gan_ckpt_dir']}/discriminator_final.pth"))

print("Loaded trained GAN.")


Loaded trained GAN.


/tmp/ipykernel_2923839/3955423846.py:5: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  gen.load_state_dict(torch.load(f"{CONFIG['gan_ckpt_dir']}/generator_final.pth"))
/tmp/i

In [7]:
# disc_feat.load_state_dict(disc.state_dict(), strict=False)
# disc_feat = disc_feat.to(CONFIG["device"]).eval()

disc_state = disc.state_dict()
remapped = {}
for key, val in disc_state.items():
    if key.startswith('7.'):
        continue
    else:
        remapped['down_layers.' + key] = val

result = disc_feat.load_state_dict(remapped, strict=False)
print("Missing:", result.missing_keys)
print("Unexpected:", result.unexpected_keys)

disc_feat = disc_feat.to(CONFIG["device"]).eval()
# Freeze generator + disc_feat
for p in gen.parameters():
    p.requires_grad = False

for p in disc_feat.parameters():
    p.requires_grad = False


Missing: ['act_final.weight', 'act_final.bias']
Unexpected: []


In [8]:
ckpt_dir = CONFIG["encoder_ckpt_dir"]
ckpts = sorted(glob.glob(os.path.join(ckpt_dir, "encoder_ep_*.pth")))

if ckpts:
    latest = ckpts[-1]
    last_epoch = int(latest.split("_ep_")[1].split(".")[0])
    print("Found previous encoder checkpoint at epoch:", last_epoch)

    enc.load_state_dict(torch.load(latest, map_location=CONFIG["device"]))
    start_epoch = last_epoch + 1
else:
    print("No previous encoder checkpoints found. Starting from scratch.")
    start_epoch = 1


Found previous encoder checkpoint at epoch: 180


/tmp/ipykernel_2923839/694482586.py:9: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  enc.load_state_dict(torch.load(latest, map_location=CONFIG["device"]))


In [9]:
trainer = EncoderTrainer(
    encoder=enc,
    gen=gen,
    disc=disc_feat,
    savefolder=CONFIG["encoder_ckpt_dir"],
    device=CONFIG["device"]
)

trainer.start = start_epoch

enc = trainer.train(
    train_data=train_loader,
    val_data=val_loader,
    epochs=CONFIG["encoder_epochs"],
    lr=CONFIG["enc_lr"],
    save_freq=10
)


/users/5/raj00075/Desktop/Anomaly_detection/KM_VERITAS_GAN/gan/trainer_encoder.py:41: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  self.scaler = GradScaler()



Epoch 181
-------------------------------------------------------


Training:   0%|          | 0/11180 [00:00<?, ?it/s]/users/5/raj00075/Desktop/Anomaly_detection/KM_VERITAS_GAN/gan/trainer_encoder.py:55: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():
Validation: 100%|██████████| 1243/1243 [01:33<00:00, 13.22it/s, img_loss: 7.10e+02 feat_loss: 1.78e+12 total_loss: 8.88e+11]



Epoch 182
-------------------------------------------------------


Validation: 100%|██████████| 1243/1243 [01:18<00:00, 15.80it/s, img_loss: 7.36e+02 feat_loss: 1.46e+12 total_loss: 7.30e+11]


...


Epoch 199
-------------------------------------------------------


Validation: 100%|██████████| 1243/1243 [01:16<00:00, 16.21it/s, img_loss: 6.36e+02 feat_loss: 1.03e+12 total_loss: 5.17e+11]



Epoch 200
-------------------------------------------------------


Validation: 100%|██████████| 1243/1243 [01:17<00:00, 16.01it/s, img_loss: 6.20e+02 feat_loss: 1.15e+12 total_loss: 5.74e+11]


Saving to /projects/standard/fortson/shared/raj00075/outputs/encoder_checkpoints_new//encoder_ep_200.pth


In [10]:
torch.save(enc.state_dict(), f"{CONFIG['encoder_ckpt_dir']}/encoder_final.pth")
print("Encoder training complete.")


Encoder training complete.
